# Неделя 03 — Скрытые пары
(i.sokolov@innopolis.university)

Эта базовая модель предсказывает, относится ли скрытая пара к положительному классу (`target = 1`), используя 512 анонимных числовых признаков.

Мы обучаем одно небольшое дерево решений на 2 520 строках обучающей выборки, оцениваем его на неизменённой валидационной выборке из 560 строк и используем тестовую выборку из 1 116 строк только для получения вероятностей. Это учебная базовая модель, а не решение, оптимизированное для лидерборда.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Чтобы использовать Colab, загрузите этот ноутбук вместе с файлами `train.csv`, `validation.csv`, `test.csv` и `sample_submission.csv`, затем запустите все ячейки. После загрузки файлов интернет и GPU не требуются.

## 1. Импорт библиотек и воспроизводимость

pandas используется для работы с CSV-таблицами, NumPy — для числовых проверок, а scikit-learn предоставляет модель и метрику ROC-AUC. PyTorch не используется, поскольку предоставленные файлы уже содержат необходимые 512 числовых признаков.

In [19]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
!mkdir /DATA_DIR
SEED = 20260916
random.seed(SEED)
np.random.seed(SEED)
notebook_started = time.perf_counter()

mkdir: cannot create directory ‘/DATA_DIR’: File exists


## 2. Загрузка предоставленных файлов

Разбиения уже подготовлены. Не создавайте другое случайное разбиение, не объединяйте выборки и не обучайте ничего на тестовых данных.

In [20]:
# DATA_DIR — каталог, содержащий четыре предоставленных CSV-файла.
DATA_DIR = Path(".")

train = pd.read_csv(DATA_DIR/ "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Обучающая выборка:", train.shape)
print("Валидационная выборка:", validation.shape)
print("Тестовая выборка:", test.shape)
print("Образец отправки:", sample_submission.shape)

Обучающая выборка: (2520, 514)
Валидационная выборка: (560, 514)
Тестовая выборка: (1116, 513)
Образец отправки: (1116, 2)


## 3. Проверка входных данных

Перед моделированием проверьте ожидаемое число строк, обязательные столбцы, пропущенные значения, бинарные целевые значения и одинаковый порядок признаков.

In [21]:
assert train.shape == (2520, 514)
assert validation.shape == (560, 514)
assert test.shape == (1116, 513)
assert sample_submission.shape == (1116, 2)

assert {"row_id", "target"}.issubset(train.columns)
assert {"row_id", "target"}.issubset(validation.columns)
assert "row_id" in test.columns and "target" not in test.columns
assert list(sample_submission.columns) == ["row_id", "target"]

assert not train.isna().any().any()
assert not validation.isna().any().any()
assert not test.isna().any().any()
assert set(train["target"].unique()) == {0, 1}
assert set(validation["target"].unique()) == {0, 1}

## 4. Выбор признаков

`row_id` используется для сопоставления строк, а `target` является ответом. Ни один из этих столбцов не является признаком модели. Во всех выборках должны присутствовать одни и те же 512 упорядоченных столбцов признаков.

### TODO(student) — Задание 1: выбрать столбцы признаков

Создайте `feature_columns` из `train.columns`, исключив `row_id` и `target`. Доступные переменные — это столбцы таблицы `train`. Результат представляет собой упорядоченные входные признаки модели и должен быть списком ровно из 512 названий.

In [26]:

feature_columns = [column for column in train.columns if column not in ("row_id", "target")]

In [27]:
assert len(feature_columns) == 512
assert not {"row_id", "target"}.intersection(feature_columns)
assert feature_columns == [column for column in validation.columns if column not in {"row_id", "target"}]
assert feature_columns == [column for column in test.columns if column != "row_id"]

## 5. Формирование входных данных модели

Не используйте валидационные данные при обучении. Дерево получает предоставленные числовые столбцы напрямую, поэтому обучаемого шага предобработки нет.

### TODO(student) — Задание 2: сформировать матрицы и целевые векторы

Используя `train`, `validation`, `test` и `feature_columns`, создайте `X_train`, `y_train`, `X_validation`, `y_validation` и `X_test`. Они представляют три матрицы признаков и два доступных целевых вектора. Каждая матрица признаков должна иметь 512 столбцов, а длина каждого целевого вектора должна соответствовать своей размеченной выборке.

In [28]:

X_train=train[feature_columns].to_numpy()
y_train=train["target"].to_numpy()
X_validation=validation[feature_columns].to_numpy()
y_validation=validation["target"].to_numpy()
X_test=test[feature_columns].to_numpy()

In [29]:
assert X_train.shape == (2520, 512)
assert X_validation.shape == (560, 512)
assert X_test.shape == (1116, 512)
assert len(y_train) == len(X_train)
assert len(y_validation) == len(X_validation)

## 6. Определение и обучение базовой модели

Дерево глубины 3 прозрачно и быстро. `min_samples_leaf=20` не позволяет строить правила по очень маленьким группам. Модель обучается только на строках обучающей выборки.

### TODO(student) — Задание 3: создать модель

Создайте `model` как `DecisionTreeClassifier` с параметрами `max_depth=3`, `min_samples_leaf=20` и `random_state=SEED`. Результатом должен быть ещё не обученный классификатор.

In [30]:

model=DecisionTreeClassifier(max_depth=3,min_samples_leaf=20,random_state=SEED)

### TODO(student) — Задание 4: обучить модель

Обучите `model`, используя только `X_train` и `y_train`. Результатом должен быть классификатор, обученный исключительно на обучающих строках и имеющий метод `predict_proba`.

In [31]:
model.fit(X_train,y_train)

DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=20260916)

## 7. Оценка на валидационной выборке

ROC-AUC показывает, насколько хорошо непрерывные оценки ранжируют положительные примеры выше отрицательных при всех возможных порогах. Классовые метки после применения одного порога теряют информацию о ранжировании, поэтому их не следует использовать для расчёта ROC-AUC.

### TODO(student) — Задание 5: валидационные вероятности и ROC-AUC

Используя `model`, `X_validation` и `y_validation`, создайте вероятности класса 1 `validation_probability` и скаляр `validation_auc`. На каждую валидационную строку должна приходиться одна вероятность, а AUC должен быть конечным и приблизительно равным 0.7473.

In [32]:
validation_probability=model.predict_proba(X_validation)[: , 1]
validation_auc=roc_auc_score(y_validation,validation_probability)

In [33]:
assert validation_probability.shape == (len(validation),)
assert np.isfinite(validation_auc)
assert np.isclose(validation_auc, 0.7473086734693878)
assert 0.0 <= validation_auc <= 1.0

## 8. Предсказание для тестовой выборки

Тестовая выборка остаётся неразмеченной и используется только после обучения. Повторное предсказание проверяет детерминированность. Ориентировочный публичный результат преподавателя равен примерно 0.72157; публичные метки недоступны этому ноутбуку и не используются в нём.

### TODO(student) — Задание 6: вероятности для тестовой выборки

Используя `model` и `X_test`, создайте вероятности класса 1 `test_probability`. Повторите предсказание и сохраните его как `test_probability_repeat`; оба массива должны быть одинаковыми и содержать по одной вероятности из диапазона `[0, 1]` для каждой тестовой строки.

In [34]:
test_probability = model.predict_proba(X_test)[:, 1]
test_probability_repeat = model.predict_proba(X_test)[:, 1]

## 9. Создание и проверка файла отправки

Использование образца отправки сохраняет требуемый порядок строк и схему. Замените только столбец `target` вероятностями положительного класса.

### TODO(student) — Задание 7: заполнить и сохранить файл отправки

Скопируйте `sample_submission` в `submission`, замените целевые значения на `test_probability`, проверьте схему из двух столбцов, число строк, точный порядок тестовых ID, уникальность ID, отсутствие пропусков и диапазон вероятностей, затем сохраните `submission.csv` с `index=False`.

In [35]:
submission = sample_submission.copy()
submission["target"] = test_probability

In [36]:
runtime_секунд = time.perf_counter() - notebook_started
print(f"ROC-AUC на валидации: {validation_auc:.6f}")
print(f"Время работы: {runtime_секунд:.2f} секунд")
print(submission.head())

ROC-AUC на валидации: 0.747309
Время работы: 1699.12 секунд
                     row_id    target
0  row_46ad1afad8562239f79c  0.865625
1  row_b42ade9cd57b8d919acb  0.865625
2  row_c951cd5e197b6b4e8b83  0.611111
3  row_a54b3423760d599416ce  0.322105
4  row_a87c87cc47690e2a303f  0.322105


## TODO(student) — Задание 8: самостоятельное сравнение моделей

Обучите и сравните дерево решений, логистическую регрессию, случайный лес и градиентный бустинг, используя предоставленное разбиение train/validation. Представьте ROC-AUC на валидации и время работы каждой модели в компактной таблице, затем обоснуйте итоговый выбор. Не обучайте и не настраивайте модели на валидационных данных.

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

candidates = {
    "Decision Tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
}

results = []
for name, clf in candidates.items():
    start = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    proba = clf.predict_proba(X_validation)[:, 1]
    auc = roc_auc_score(y_validation, proba)

    results.append({"Модель": name, "ROC-AUC (валидация)": round(auc, 4), "Время (сек)": round(fit_seconds, 3)})

comparison_table = pd.DataFrame(results).sort_values("ROC-AUC (валидация)", ascending=False).reset_index(drop=True)
print(comparison_table)

                Модель  ROC-AUC (валидация)  Время (сек)
0    Gradient Boosting               0.9051       73.839
1        Random Forest               0.9032       17.640
2  Logistic Regression               0.8903        1.287
3        Decision Tree               0.7473        1.905


Среди четырех моделей наилучший показатель ROC-AUC на валидации выдал Decision tree при чуть дольшем времени обучения чем лог регрессия.Исходя из этого выбираем модель Decision tree
